# Suite PAT — Patterns: symbols, variables, constraints, locators, expressions and collectors

A pattern (`Schemas.OfPattern`) is the shape of a frame (docs/STREAMING_QUERIES.md): its symbols, each standing for a
reference object, its variables, each standing for a value, and the constraints a frame meets, expressions of boolean
result.
An expression (`Schemas.OfExpression`) ranges over a pattern; its terms are mbse-expressions'. A collector
(`Schemas.OfCollector`) folds a pattern's frames. Each is a schema object, built, validated and written as every one is;
frames, and matching, are the executor's (step 4e of the store overhaul).

In [ ]:
from mbse.Schemas.Framework import Builders as B, JSON, Plain, Schemas as S
from support import raises


def wts(txn):
    """wts.Features, a singleton directory owning definitions through `dir`, keyed by name, each owning its clauses
    through `clauses`."""
    S = txn.Schemas
    return txn.uid("wts/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Directory": S.OfObject, "Definition": S.OfObject, "Clause": S.OfObject, "Defines": S.OfRelation,
                  "HasClause": S.OfRelation})
        .spec([
            lambda _: _.Directory.name("wts.Directory").ref(True).singleton("wts.Features").adjacencies([
                lambda t: t.name("dir").relation([_.Defines]).me("directory")]),
            lambda _: _.Definition.name("wts.Definition").ref(True).properties([
                lambda t: t.name("title").schema([S.OfStr]),
            ]).adjacencies([
                lambda t: t.name("def").relation([_.Defines]).me("defn").owner(True),
                lambda t: t.name("clauses").relation([_.HasClause]).me("defn"),
            ]),
            lambda _: _.Clause.name("wts.Clause").ref(True).adjacencies([
                lambda t: t.relation([_.HasClause]).me("clause").owner(True)]),
            lambda _: _.Defines.name("wts.Defines").links([lambda t: t.name("directory"), lambda t: t.name("defn")]).properties([
                lambda t: t.name("name").schema([S.OfStr])]).uniques([["directory", "name"], ["defn"]]),
            lambda _: _.HasClause.name("wts.HasClause").links([lambda t: t.name("defn"), lambda t: t.name("clause")]).properties([
                lambda t: t.name("name").schema([S.OfStr])]).uniques([["defn", "name"], ["clause"]]),
        ])]))])


store = B.Store()
Text, Int, Bool = (store.Schemas.OfStr.as_schema(), store.Schemas.OfInt.as_schema(), store.Schemas.OfBool.as_schema())
store.transaction(wts)
Definition, Clause = store.registered("wts.Definition"), store.registered("wts.Clause")

## PAT-01 · A pattern: symbols, variables and constraints, in order, each its own; validated

In [ ]:
def edits(txn):
    """wts.Edit, a pattern over what `wts` installed."""
    S = txn.Schemas
    return txn.uid("edit/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Edit": S.OfPattern})
        .spec([lambda _: (
            _.Edit
            .name("wts.Edit")
            .description("A definition and one of its clauses")
            .symbols([
                lambda t: t.name("defn").schema([txn.wts.Definition]),
                lambda t: t.name("clause").schema([txn.wts.Clause]).description("C-1"),
            ])
            .variables([lambda t: t.name("title").schema([S.OfStr]).description("its new title")])
            .requires([lambda c: c.description("the title is new")])  # a constraint: its result, OfBool, by default
        )])]))])


store.transaction(edits)
Edit = store.registered("wts.Edit")
assert Edit.validate() == [] and list(Edit.symbols) == ["defn", "clause"] and list(Edit.variables) == ["title"]
assert Edit.variables["title"].schema is Text and Edit.symbols["clause"].description == "C-1"
assert len(Edit.requires) == 1 and Edit.requires[0].result is Bool  # a constraint: an expression of the store's OfBool
assert store.registered("wts.Edit") is Edit and store.module("wts").schemas["Edit"] is Edit  # named: in its module
with store.transaction():
    twin = B.OfPattern.Builder(Edit, True, store).name("wts.Twin").clone()
    assert twin.symbols["defn"] is not Edit.symbols["defn"] and twin.requires[0] is not Edit.requires[0]  # a clone owns its own
    assert twin.variables["title"] is not Edit.variables["title"] and twin.variables["title"].schema is Text
    Port = store.Schemas.OfObject(lambda o: o.properties([lambda t: t.name("pin").schema([Text])]))
    bad = store.Schemas.OfPattern(lambda p: (
        p
        .symbols([lambda t: t.name("x").schema([Port]), lambda t: t.name("v")])
        .variables([lambda t: t.name("v").schema([Definition]), lambda t: t.name("w")])
    ))
    bad.variables["w"].description = 1  # what the builders refuse
    assert bad.validate() == [
        "names used as both symbol and variable: ['v']",
        "symbol 'x': a symbol stands for a reference object, so its schema is a reference object schema",
        "variable 'v': a reference object schema cannot be a property's schema",
        "variable 'w': a description is text, got int"]
assert S.OfPattern.Data(symbols={"p": S.OfSymbol.Data("p")}, variables={"p": S.OfVariable.Data("p")}).validate() == [
    "names used as both symbol and variable: ['p']"]

## PAT-02 · Expressions and constraints: a pattern and a result schema; a constraint's result is OfBool; a pattern requires its own constraints

In [ ]:
def counts(txn):
    """wts.Count, an expression over wts.Edit."""
    S = txn.Schemas
    return txn.uid("count/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Count": S.OfExpression})
        .spec([lambda _: _.Count.name("wts.Count").pattern([txn.wts.Edit]).result([S.OfInt])])]))])


store.transaction(counts)
Count = store.registered("wts.Count")
assert Count.validate() == [] and Count.pattern is Edit and Count.result is Int
assert store.registered("wts.Count") is Count and Count == Count and Count != Edit
with store.transaction():
    Holds = store.Schemas.OfConstraint(lambda c: c.name("wts.Holds").pattern([Edit]))  # a builder's kind: its result fixed
    assert Holds.result is Bool and Holds.validate() == [] and isinstance(Holds, S.OfExpression.Data)
    assert B.OfConstraint.Builder().create().result.name == "Schemas.OfBool"  # outside a store, the template's
    assert S.OfExpression.Data(pattern=Count).validate() == ["an expression's pattern is a pattern", "an expression needs a result schema"]
    assert S.OfExpression.Data(result=Port, name="x y").validate() == [
        "name 'x y' is not identifiers separated by dots, e.g. 'crm.Contact'"]
    assert S.OfExpression.Data(result=Definition).validate() == ["result: a reference object schema cannot be a property's schema"]
    own = S.OfExpression.Data(result=Bool)
    own.pattern = mine = S.OfPattern.Data(requires=(own,))
    assert mine.validate() == []  # over its own pattern
    assert S.OfPattern.Data(requires=(Count,)).validate() == [
        "requires[0] is a constraint: an expression whose result is Schemas.OfBool"]
    assert S.OfPattern.Data(requires=(Holds,)).validate() == ["requires[0] is over this pattern's frames, not another's"]
    assert S.OfPattern.Data(requires=(S.OfExpression.Data(result=Bool, description=2),)).validate() == [
        "requires[0]: a description is text, got int"]

## PAT-03 · Collectors: a pattern, the names it collects, commutative or associative, two suffixes, and what it computes

In [ ]:
def sums(txn):
    """wts.Sum, a collector of wts.Edit's frames, computing wts.Count."""
    return txn.uid("sum/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Sum": txn.Schemas.OfCollector})
        .spec([lambda _: (
            _.Sum
            .name("wts.Sum")
            .pattern([txn.wts.Edit])
            .symbols(["clause"])
            .variables(["title"])
            .commutative(True)
            .associative(True)
            .suffixes({"current": "_a", "next": "_b"})
            .computes([txn.wts.Count])
        )])]))])


store.transaction(sums)
Sum = store.registered("wts.Sum")
with store.transaction():
    assert Sum.validate() == [] and (Sum.symbols, Sum.variables, Sum.suffixes) == (("clause",), ("title",), {"current": "_a", "next": "_b"})
    assert B.OfCollector.Builder(Sum, False, store).symbols._held() == ["clause"] and Sum.computes is Count and Sum == Sum
    cleared = B.OfCollector.Builder(Sum, True, store).name("wts.Cleared").symbols(None).variables(None).suffixes(None).clone()
    assert (cleared.symbols, cleared.variables, cleared.suffixes) == ((), (), None)
    assert S.OfCollector.Data().validate() == [
        "a collector needs a pattern",
        "a collector is commutative or associative, so that its result does not depend on the order frames arrive in"]
    assert S.OfCollector.Data(Edit, ("nope",), ("none",), associative=True).validate() == [
        "it collects symbols ['nope'], which its pattern does not declare",
        "it collects variables ['none'], which its pattern does not declare"]
    assert S.OfCollector.Data(Edit, commutative=True, suffixes={"current": "x"}).validate() == [
        "a collector's suffixes are 'current' and 'next'"]
    assert S.OfCollector.Data(Edit, commutative=True, suffixes={"current": "x", "next": "x"}).validate() == [
        "a collector's 'current' and 'next' suffixes differ"]
    assert S.OfCollector.Data(Edit, commutative=True, computes=Edit).validate() == ["what a collector computes is an expression"]
    assert S.OfCollector.Data(Edit, commutative=True, computes=S.OfExpression.Data()).validate() == [
        "computes: an expression needs a result schema"]
    assert B.OfCollector.Builder(store.Schemas.OfCollector(lambda c: c), False, store).suffixes._held() is None

## PAT-04 · Patterns, expressions and collectors in a store's snapshot: written once, read back the same

In [ ]:
log = Plain.ToPlain(store).Snapshot()
back = B.Store()
Plain.FromPlain(back).Snapshot(log)
assert Plain.ToPlain(back).Snapshot() == log
assert JSON.loads(JSON.ToJSON(back).Snapshot()) == log
read = back.registered("wts.Edit")
assert read is not Edit and list(read.symbols) == ["defn", "clause"] and read.requires[0].result is back.Schemas.OfBool.as_schema()
assert back.registered("wts.Sum").pattern is read and back.registered("wts.Sum").computes is back.registered("wts.Count")